# Software profesional en Acústica 2025-26 (M2i)

*This notebook was adapted from Chapter 1 of [The FEniCS Tutorial Volume I](https://fenicsproject.org/pub/tutorial/sphinx1/) by Hans Petter Langtangen and Anders Logg, released under CC Attribution 4.0 license. It has been created by Xiangmin Jiao (University of Stony Brook University) and it is available in the repository [Unifem/FEniCS-note](https://github.com/unifem/fenics-notes). The FEniCS implementation has been replaced with NGSolve.*

First, we need to install on the fly [NGSolve](https://ngsolve.org/) using the [FEM on Colab](https://fem-on-colab.github.io/packages.html) install script:

In [1]:
try:
    import ngsolve
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/ngsolve-install-release-complex.sh" -O "/tmp/ngsolve-install.sh" && bash "/tmp/ngsolve-install.sh"
    import ngsolve

# The equations of vibrations in linear elasticity

Vibrations in linear elasticity is the study of how solid objects are responding to a mechanical vibration and become 
internally stressed due to prescribed time-harmonic loading conditions. It is an important problem
in modern engineering. Its corresponding PDE is a generalization of the
Helmholtz equation, and it is among one of the most popular PDEs in 
engineering. We now study its variational formulation and how to solve
this problem using NGSolve in 2D.

## PDE problem

The time-harmonic equation governing elastic vibrations of a body $\Omega$ can be written as

\begin{equation}
\tag{1}
-\omega^2\rho\boldsymbol{u} -\mathrm{div}\,\boldsymbol{\sigma} = \boldsymbol{f}\hbox{ in }\Omega,
\end{equation}

where $\boldsymbol{\sigma}$ is the *stress tensor*, and $\boldsymbol{f}$ is the *body force per unit
volume*, $\rho$ is the *mass density*, and $\omega$ the angular frequency. For isotropic materials, the stress tensor is further related to the deformation by 
the following two equations:
\begin{align}
\boldsymbol{\sigma} &= \lambda\,\hbox{tr}\,(\boldsymbol{\varepsilon}) \boldsymbol{I} + 2\mu\boldsymbol{\varepsilon},
\tag{2}\\
\boldsymbol{\varepsilon} &= \frac{1}{2}\left(\boldsymbol{\nabla} \boldsymbol{u} + (\boldsymbol{\nabla} \boldsymbol{u})^{\top}\right),
\tag{3}
\end{align}
where $\boldsymbol{\varepsilon}$ is the *symmetric strain-rate tensor* (symmetric gradient), 
and $\boldsymbol{u}$ is the *displacement vector field*, $\boldsymbol{I}$ denotes the *identity tensor*, 
$\mathrm{tr}$ denotes the *trace operator* on a tensor, and $\lambda$ and $\mu$ 
are material properties known as *Lamé's elasticity parameters*.

We can combine (2)-(3) to obtain
\begin{equation}
\tag{4}
\boldsymbol{\boldsymbol{\sigma}} = \lambda(\boldsymbol{\nabla}\cdot \boldsymbol{u})\boldsymbol{I} + \mu(\boldsymbol{\nabla} \boldsymbol{u} + (\boldsymbol{\nabla} \boldsymbol{u})^{\top})
\end{equation}

Note that (1) and (4) can easily be transformed to a single vector PDE for $\boldsymbol{u}$, which is the
governing PDE for the unknown $\boldsymbol{u}$ (Navier's equation).  In the
derivation of the variational formulation, however, it is convenient
to keep the equations split as above.

## Variational formulation

The variational formulation of (1) and (4) consists of forming the inner product of
(1) and a *vector* test function $\boldsymbol{v}\in \hat{V}$, where $\hat{V}$ is a vector-valued test function space, and
integrating over the domain $\Omega$:

\begin{equation} 
-\omega^2\int_\Omega \rho\boldsymbol{u}\cdot \boldsymbol{v}\ \mathrm{d}\boldsymbol{x}-\int_\Omega (\mathrm{div}\,\boldsymbol{\sigma}) \cdot \boldsymbol{v}\ \mathrm{d}\boldsymbol{x} =
\int_\Omega \boldsymbol{f}\cdot \boldsymbol{v}\ \mathrm{d}\boldsymbol{x},\tag{5}
\end{equation}
where $\mathrm{d}\boldsymbol{x}$ denotes the volume measure.

Since $\mathrm{div}\,\boldsymbol{\sigma}$ contains second-order derivatives of the primary
unknown $\boldsymbol{u}$, we integrate this term by parts:
\begin{equation}
-\omega^2\int_\Omega \rho\boldsymbol{u}\cdot \boldsymbol{v}\ \mathrm{d}\boldsymbol{x}-\int_\Omega (\mathrm{div}\,\boldsymbol{\sigma}) \cdot \boldsymbol{v} \ \mathrm{d}\boldsymbol{x}
= \int_\Omega \boldsymbol{\sigma} : \boldsymbol{\nabla} \boldsymbol{v} \ \mathrm{d}\boldsymbol{x} - \int_{\partial\Omega}
(\boldsymbol{\sigma}\boldsymbol{n})\cdot \boldsymbol{v} \ \mathrm{d}\boldsymbol{s},\tag{6}
\end{equation}

where the colon operator is the inner product between tensors (summed
pairwise product of all elements), $\boldsymbol{n}$ is the outward unit normal
at the boundary, and $\mathrm{d}\boldsymbol{s}$ is a measure in surface area.

The quantity $\boldsymbol{\sigma}\boldsymbol{n}$ is known as the
*traction* or stress vector at the boundary, and is often prescribed
as a boundary condition on a part $\Gamma_{\mathrm{T}}\subset\partial\Omega$ of the boundary 
as $\boldsymbol{\sigma}\boldsymbol{n} = \boldsymbol{g}_{\mathrm{T}}$, whereas the 
remaining part of the boundary either the value of the displacements is prescribed or a free traction boundary is considered.

Substituting (5) into (6), we thus obtain

\begin{equation}
-\omega^2\int_\Omega \rho\boldsymbol{u}\cdot \boldsymbol{v}\ \mathrm{d}\boldsymbol{x}
+ \int_\Omega \boldsymbol{\sigma} : \boldsymbol{\nabla} \boldsymbol{v}\ \mathrm{d}\boldsymbol{x} =
\int_\Omega \boldsymbol{f}\cdot \boldsymbol{v}\ \mathrm{d}\boldsymbol{x}
+ \int_{\Gamma_{\mathrm{T}}} \boldsymbol{g}_\mathrm{T}\cdot \boldsymbol{v}\ \mathrm{d}\boldsymbol{s}.\tag{7}
\end{equation}

Inserting the expression (4) for
$\boldsymbol{\sigma}$ gives the variational form with $\boldsymbol{u}$ as unknown. 

### Symmetrizing $\boldsymbol{\nabla} \boldsymbol{v}$

One can show that the inner product of a symmetric tensor $\boldsymbol{A}$ and an
anti-symmetric tensor $\boldsymbol{B}$ vanishes. Since $\boldsymbol{\sigma}$ is a
symmetric tensor, if we express $\boldsymbol{\nabla} \boldsymbol{v}$ as a sum
of its symmetric and anti-symmetric parts, only the symmetric part will
survive in the product $\boldsymbol{\sigma} :\boldsymbol{\nabla} \boldsymbol{v}$. Thus replacing $\boldsymbol{\nabla} \boldsymbol{u}$ by the symmetric gradient gives rise to the slightly different variational form

\begin{equation}
-\omega^2\int_\Omega \rho\boldsymbol{u}\cdot \boldsymbol{v}\ \mathrm{d}\boldsymbol{x}
+ \int_\Omega \boldsymbol{\sigma} : \boldsymbol{\epsilon}(\boldsymbol{v})\ \mathrm{d}\boldsymbol{x} =
\int_\Omega \boldsymbol{f}\cdot \boldsymbol{v}\ \mathrm{d}\boldsymbol{x}
+ \int_{\Gamma_\mathrm{T}} \boldsymbol{g}_\mathrm{T}\cdot \boldsymbol{v}\ \mathrm{d}\boldsymbol{s}
\tag{8}
\end{equation}

where $\boldsymbol{\epsilon}(\boldsymbol{v})$ is the symmetric part of $\boldsymbol{\nabla} \boldsymbol{v}$:
\begin{equation}
\boldsymbol{\epsilon}(\boldsymbol{v}) = \frac{1}{2}\left(\boldsymbol{\nabla} \boldsymbol{v} + (\boldsymbol{\nabla} \boldsymbol{v})^{\top}\right).
\tag{9}
\end{equation}

The formulation (\ref{ftut-elast-varform-sigma_inner_gradv}) is what naturally
arises from the minimization of elastic potential energy, and is a more
popular formulation than (7).

Here, $\boldsymbol{\epsilon}$ is a useful operator. The symmetric strain-rate tensor $\boldsymbol{\varepsilon}$ in (3) is equal to $\boldsymbol{\epsilon}(\boldsymbol{u})$. 

### Enforcing boundary conditions

Now let us consider how to enforce boundary conditions. 
For Dirichlet boundaries, we will enforce boundary-conditions strongly.
For these points, no test functions are associated with the Dirichlet nodes.

For traction boundary conditions, we will enforce the boundary condition
weakly using the variational form (8).
Similar to the Helmholtz equation, we require their corresponding test
functions $\boldsymbol{v}$ vanish along $\partial \Omega$ for interior points.
Then, the boundary integral above has no effects for points on
$\partial\Omega\setminus\Gamma_\mathrm{T}$.

### Summary of variational form
In summary, the variational problem is to find $\boldsymbol{u}$ in a vector function space $\hat{V}$ such that

\begin{equation}
a(\boldsymbol{u},\boldsymbol{v}) = L(\boldsymbol{v})\quad\forall \boldsymbol{v}\in\hat{V},\tag{10}
\end{equation}
where 

\begin{align}
a(\boldsymbol{u},\boldsymbol{v}) &= -\omega^2\int_\Omega \rho\boldsymbol{u}\cdot \boldsymbol{v}\ \mathrm{d}\boldsymbol{x}
+ \int_\Omega\sigma(\boldsymbol{u}) :\varepsilon(\boldsymbol{v})\ \mathrm{d}\boldsymbol{x},\tag{11}\\
L(\boldsymbol{v}) &= \int_\Omega \boldsymbol{f}\cdot \boldsymbol{v}\ \mathrm{d}\boldsymbol{x} + \int_{\Gamma_{\mathrm{T}}} \boldsymbol{g}_{\mathrm{T}}\cdot \boldsymbol{v}\ \mathrm{d}\boldsymbol{s}\tag{12},
\end{align}
and 

\begin{equation}
\boldsymbol{\sigma}(\boldsymbol{u}) = \lambda(\boldsymbol{\nabla}\cdot \boldsymbol{u})\boldsymbol{I} + \mu(\boldsymbol{\nabla} \boldsymbol{u} + (\boldsymbol{\nabla} \boldsymbol{u})^{\top}).\tag{13}\\
\end{equation}

## NGSolve implementation

To demonstrate the implementation, we will model a clamped beam deformed under a time-harmonic surface force on the opposite free cross-section surface. This can be modeled by setting $\boldsymbol{g}_{\mathrm{T}}=(0,1)$ on that boundary part $x=L$. The beam is
box-shaped with length $L$ and has a square cross section of width $W$. We
set $\boldsymbol{u}=(0,0)$ on the clamped end, $x=0$. The rest of the lateral boundary is
traction free (so, no other boundary contributions are involved in the variational formulation). Therefore,
$$L(\boldsymbol{v}) = \int_{\Gamma_{\mathrm{T}}} \boldsymbol{g}_{\mathrm{T}}\cdot \boldsymbol{v} \mathrm{d}\boldsymbol{s}$$
for this problem.

We use **complex arithmetic** throughout (via `complex=True` in the function space definition), even though the data is real-valued, for consistency with other time-harmonic formulations.

### Import packages

We start by importing NGSolve and matplotlib.

In [2]:
import numpy as np
from ngsolve import *
from netgen.geom2d import SplineGeometry
from ngsolve.webgui import Draw

### Generate the mesh and function spaces

Our action starts by generating the mesh and defining function spaces.
We label the four sides of the rectangle:
- `"left"` — the clamped end ($x=0$), where $\boldsymbol{u}=\boldsymbol{0}$
- `"right"` — the traction end ($x=L$), where $\boldsymbol{g}_T=(0,1)$ is applied
- `"bottom"` and `"top"` — traction-free lateral sides

In [3]:
# Geometry
length = 1.0
width  = 0.2

geo = SplineGeometry()
geo.AddRectangle(
    (0, 0), (length, width),
    bcs=["bottom", "right", "top", "left"]
)
ngmesh = geo.GenerateMesh(maxh=0.02)
mesh = Mesh(ngmesh)

# Compute hmax from mesh vertex coordinates
coords = np.array([[v.point[0], v.point[1]] for v in mesh.vertices])
xmin, ymin = coords.min(axis=0)
xmax, ymax = coords.max(axis=0)
# hmax: maximum edge length (approximate via bounding box diagonal / sqrt(#elements))
num_els = mesh.ne
hmax = np.sqrt((xmax - xmin)**2 + (ymax - ymin)**2) / np.sqrt(num_els)
print(f"Mesh: {mesh.nv} vertices, {num_els} elements, hmax ≈ {hmax:.4f}")

# Vector H1 space with complex arithmetic; Dirichlet BC on the left edge
fes = VectorH1(mesh, order=1, dirichlet="left", complex=True)
print(f"Number of DOFs: {fes.ndof}")

# Plot mesh
Draw(mesh, height="3vh")

Mesh: 631 vertices, 1140 elements, hmax ≈ 0.0302
Number of DOFs: 1262


WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.260…

BaseWebGuiScene

### Define the variational problem

The primary unknown is now a vector field $\boldsymbol{u}$ and not a scalar field,
so we need to work with a vector function space. We will use 
piecewise-linear basis functions for all the components.


In [4]:
u, v = fes.TnT()   # trial and test functions

Next, we define the strain tensor $\boldsymbol{\varepsilon}$, the stress tensor $\boldsymbol{\sigma}$, and the bilinear form $a$.
In NGSolve the symmetric gradient is available via `Sym(Grad(u))`.

In [5]:
# Physical constants
rho     = 1.0
omega   = 2 * np.pi * 3       # angular frequency
beta    = 1.25
lambda_ = beta                # first Lamé parameter
mu      = 1.0                 # second Lamé parameter (shear modulus)

# Symmetric strain tensor  ε(u) = ½(∇u + (∇u)ᵀ)
def epsilon(w):
    return Sym(Grad(w))

# Stress tensor  σ(u) = λ(∇·u)I + 2με(u)
def sigma(w):
    return lambda_ * div(w) * Id(mesh.dim) + 2 * mu * epsilon(w)

# Bilinear form  a(u,v) = -ω²ρ∫u·v dx + ∫σ(u):ε(v) dx
a = BilinearForm(fes)
a += (-omega**2 * rho * InnerProduct(u, v) + InnerProduct(sigma(u), epsilon(v))) * dx


To define $L$, $\boldsymbol{g}_T=(0,1)$ is a constant vector applied on the right boundary.

In [6]:
# Traction force g_T = (0, 1) applied on the right boundary
g_T = CoefficientFunction((0.0, 1.0))

# Linear form  L(v) = ∫_{Γ_T} g_T · v ds
l = LinearForm(fes)
l += InnerProduct(g_T, v) * ds("right")

### Define boundary conditions

We only specify the Dirichlet boundary condition (zero displacement) on the clamped left edge.
This is already encoded in the `dirichlet="left"` argument of `VectorH1`.

### Solve the variational problem

Finally, we assemble and solve the linear system.

In [7]:
# Assemble
a.Assemble()
l.Assemble()

# Solve
gfu = GridFunction(fes)
gfu.Set(CF((0.0, 0.0)), definedon=mesh.Boundaries("left"))
gfu.vec.data = a.mat.Inverse(fes.FreeDofs()) * l.vec

### Plot the solution

We extract the real parts of the two displacement components and visualise them with matplotlib.
Even though the data is real-valued, we use `real` to obtain the physical displacement from the complex representation.

In [8]:
# First component of the solution (x-displacement)
Draw(gfu[0], mesh, animate_complex=True, height="3vh")
# Second component of the solution (y-displacement)
Draw(gfu[1], mesh, animate_complex=True, height="3vh")
# Total displacement magnitude
Draw(gfu, mesh, deformation=0.1*gfu.real, height="3vh")
# Real part of the solution with vectors
Draw(gfu.real, mesh, deformation=False, vectors={"grid_size":40, "offset": 1}, height="3vh")


WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 'spe…

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 'spe…

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 'spe…

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.260…

BaseWebGuiScene

**Copyright**

This notebook is provided as [Open Educational Resource](https://en.wikipedia.org/wiki/Open_educational_resources). Feel free to use the notebook for your own purposes. The text is licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/), the code of the IPython examples under the [MIT license](https://opensource.org/licenses/MIT).